<a href="https://colab.research.google.com/github/DanilRodenko/LexIQ/blob/main/models/03_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

df_train = pd.read_parquet('df_train.parquet')
df_train.head()

In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForQuestionAnswering
import torch

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
model = DistilBertForQuestionAnswering.from_pretrained('distilbert-base-uncased')


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
device = torch.device('cuda')
model.to(device)

In [ ]:
from torch.utils.data import Dataset

class CUADDataset(Dataset):

  def __init__(self, processed):
    self.processed = processed

  def __len__(self):
    return len(self.processed)

  def __getitem__(self, idx):
      processed = self.processed[idx]
      input_ids = list(processed['input_ids'])[:512]
      attention_mask = list(processed['attention_mask'])[:512]
      input_ids = torch.tensor(input_ids + [0] * (512 - len(input_ids)))
      attention_mask = torch.tensor(attention_mask + [0] * (512 - len(attention_mask)))
      start_position = torch.tensor(processed['start_position'])
      end_position = torch.tensor(processed['end_position'])

      return input_ids, attention_mask, start_position, end_position


In [ ]:
train_dataset = CUADDataset(df_train['processed'].values)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)

In [ ]:
from tqdm import tqdm

epochs = 3
losses = []

model.train()
for epoch in range(epochs):
    epoch_loss = 0
    loop = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch}")
    for batch_idx, (input_ids, attention_mask, start_position, end_position) in loop:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        start_position = start_position.to(device)
        end_position = end_position.to(device)

        outputs = model(input_ids, attention_mask=attention_mask, start_positions=start_position, end_positions=end_position)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        loop.set_postfix(loss=round(loss.item(), 4))

    avg_loss = round(epoch_loss / len(train_loader), 4)
    losses.append(avg_loss)
    print(f"Epoch {epoch} | Avg Loss: {avg_loss}")

In [ ]:
model.save_pretrained('./lexiq_model')
tokenizer.save_pretrained('./lexiq_model')

In [ ]:
from google.colab import files
import shutil

shutil.make_archive('lexiq_model', 'zip', './lexiq_model')
files.download('lexiq_model.zip')